In [8]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, output_guardrail, GuardrailFunctionOutput

from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [4]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Dennis
"""
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)


In [5]:
@output_guardrail
async def email_guardrail(cntxt, agent, message):
    result = await Runner.run(checker, message, context=cntxt.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [6]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model='gpt-5.4-mini', output_guardrails=[email_guardrail])

In [7]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire